In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [6]:


# Image transformations
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# CNN Model Architecture
class HumanDetectionCNN(nn.Module):
    def __init__(self):
        super(HumanDetectionCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = HumanDetectionCNN().to(device)
print("Model successfully created!")
print(model)


Model successfully created!
HumanDetectionCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [8]:
import numpy as np
from PIL import Image

# 1-1 dummy random images create karke folders me save kar rahe hain
img_array = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
img = Image.fromarray(img_array)

# Save dummy images
img.save('data/train/human/sample_human.jpg')
img.save('data/train/non_human/sample_non_human.jpg')
img.save('data/val/human/sample_human.jpg')
img.save('data/val/non_human/sample_non_human.jpg')

print("✅ Dummy sample images successfully add ho gayi hain!")

✅ Dummy sample images successfully add ho gayi hain!


In [9]:
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Automatic folder structure create karne ke liye (agar nahi hai toh)
os.makedirs('data/train/human', exist_ok=True)
os.makedirs('data/train/non_human', exist_ok=True)
os.makedirs('data/val/human', exist_ok=True)
os.makedirs('data/val/non_human', exist_ok=True)

# 2. Image transformations
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Dataset load karne ka secure tareeqa
dataset_path = 'data'

train_dataset = datasets.ImageFolder(root=os.path.join(dataset_path, 'train'), transform=transform)
val_dataset = datasets.ImageFolder(root=os.path.join(dataset_path, 'val'), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print("✅ Dataset successfully loaded!")
print("Classes found:", train_dataset.classes)
print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")

✅ Dataset successfully loaded!
Classes found: ['human', 'non_human']
Total training images: 2
Total validation images: 2


In [10]:
import torch.nn as nn
import torch.optim as optim

# 1. CNN Model Class (Agar pehle define nahi kiya hai toh yahan le sakte hain)
class HumanDetectionCNN(nn.Module):
    def __init__(self):
        super(HumanDetectionCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1) # Binary output ke liye 1 neuron
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Model ko device (CPU) par bhejna
model = HumanDetectionCNN().to(device)

# 2. Loss function aur Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Training Loop
epochs = 3  # Abhi testing ke liye 3 epochs rakhte hain

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass & Optimizer step
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        # Accuracy calculate karein
        preds = torch.sigmoid(outputs) >= 0.5
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

# 4. Model Save Karna
torch.save(model.state_dict(), 'human_detection_pytorch.pth')
print("🎉 Model successfully train hokar save ho gaya hai!")

Epoch [1/3], Loss: 0.7075, Accuracy: 0.5000
Epoch [2/3], Loss: 1.0148, Accuracy: 0.5000
Epoch [3/3], Loss: 1.9871, Accuracy: 0.0000
🎉 Model successfully train hokar save ho gaya hai!


In [11]:
import torch
from PIL import Image
from torchvision import transforms

# 1. Model architecture ko dubara initialize karein
model = HumanDetectionCNN().to(device)

# 2. Saved weights ko model me load karein
model.load_state_dict(torch.load('human_detection_pytorch.pth'))
model.eval() # Evaluation mode on karein

# 3. Test image ke liye transform (wahi jo training me use kiya tha)
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_image(image_path):
    # Image load karein
    image = Image.open(image_path).convert('RGB')
    input_tensor = test_transform(image).unsqueeze(0).to(device) # Batch dimension add karna
    
    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.sigmoid(output).item() # Probability calculate karein (0 se 1 ke beech)
        
    # Threshold 0.5 (Agar > 0.5 hai toh Human, warna Non-Human)
    predicted_class = "human" if prob >= 0.5 else "non_human"
    confidence = prob if prob >= 0.5 else (1 - prob)
    
    print(f"📁 Image: {image_path}")
    print(f"🤖 Prediction: {predicted_class.upper()} (Confidence: {confidence * 100:.2f}%)")

# Example test (Aap apne data folder ki kisi image ka path de sakte hain)
# predict_image('data/train/human/sample_human.jpg')

In [12]:
# Function ko call karke sample image test karein
predict_image('data/train/human/sample_human.jpg')

📁 Image: data/train/human/sample_human.jpg
🤖 Prediction: NON_HUMAN (Confidence: 58.11%)
